# System info

**Distro:** Linux Fedora 43 for Workstation x86-64 <br>
**Kernel:** Linux 7.0.9-105.fc43.x86_64 <br><br>

**GPU:** NVIDIA Blackwell GB203 <br>
**CPU:** Intel 12th Gen. i5-12600K @4.90 GHz <br><br>

**NVIDIA CUDA:** 12.8 <br>
**NVIDIA driver:** 580.159.03 <br>
**Pytorch:** 2.11.0 <br>
**Python:** 3.13.3 running in `conda` environment

<br><br><br>

Currently running kernel:

In [36]:
!uname -r

7.0.9-105.fc43.x86_64


In [37]:
!which python

/home/simon/anaconda3/envs/ml_environment_cuda128/bin/python


<br><br><br>

# Config

In [31]:
import torch #supports CUDA 12.8 for NVIDIA Blackwell

# custom class for config

class Config():

    #data
    dataset: str = "stanfordnlp/imdb"
    test_size: float = 0.2
    fixed_seed: int = 42
    labels: str = "label"
    texts: str = "text"
    
    #model
    task: str = "text-classification"
    model: str = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
    trunc: bool = True
    batch_size: int = 64

    #device = NVIDIA Blackwell GB203
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    



In [34]:
cfg = Config()

#set device as torch device
cfg.device = torch.device(cfg.device)

#verify
print(cfg.device)
print(torch.version.cuda)

cuda
12.8


# Dataset

In [19]:
from datasets import load_dataset

In [20]:
raw_dataset = load_dataset(cfg.dataset)

In [21]:
train_texts_final = raw_dataset["train"][cfg.texts]
train_labels_final = raw_dataset["train"][cfg.labels]

from sklearn.model_selection import train_test_split

test_texts = list(raw_dataset["test"][cfg.texts])
test_labels = list(raw_dataset["test"][cfg.labels])


# Since we don't need the larger dataset, we can ignore its output by assigning it to `_` (nothing)
_, test_texts_final, _, test_labels_final = train_test_split(
    test_texts, test_labels,  # the data to split
    test_size = cfg.test_size,    # the number of samples in the new dataset
    stratify = test_labels,      # to make sure the split is stratified based on the labels
    random_state = cfg.fixed_seed         # for reproducibility
)

In [ ]:
#import time

#use perf_counter for higher accuracy
from time import perf_counter

# Batching at BF16 (GPU)

In [6]:
from transformers import pipeline
import pandas as pd
import torch
from time import perf_counter
from sklearn.metrics import f1_score, classification_report

#load pipeline
pipe = pipeline(
    task = cfg.task,
    model = cfg.model,
    device = cfg.device,
    dtype = torch.bfloat16 #half precision
)

#different batch sizes
batches = [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]

#used for logging
history = []

for batch in batches:

    #number of batches
    if len(test_texts_final) % batch == 0:
        batch_number = len(test_texts_final)//batch
    else:
        batch_number = (len(test_texts_final)//batch) + 1
    
    #run model if it fits in video memory
    try:

        #start performance measuring
        start = perf_counter()
    
        results = pipe(
            test_texts_final,
            batch_size = batch,
            truncation = cfg.trunc
        )

        #stop time measurement                    
        end = perf_counter()

        #write results to dataframe
        df_results = pd.DataFrame(results)

        #recode label
        df_results["label"] = df_results["label"].replace(
            ["POSITIVE",
            "NEGATIVE"],
    
            [1,
            0]
        )
        

        #log times
        history.append({
            "batch_size": batch,
            "time": end - start})

        print(f"At batch size", batch, f"a total of", batch_number, f" batches were created. \n \n", 
              f"This task took {end - start: .4f} seconds with half precision (BF16) \n \n on a NVIDIA Blackwell GB203 GPU. \n \n Therefore, on average {len(test_texts_final)//(end - start): .2f} texts were classified per second. \n \n \n")

    
    #if video memory runs out, print explanation and then stop
    except torch.cuda.OutOfMemoryError:
        
        print(f"Note: At batch size", batch, f"the GPU ran out of memory.")
        
        break



#build dataframe with timings
df_history_bf16 = pd.DataFrame(history)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

At batch size 2 a total of 2500  batches were created. 
 
 This task took  8.9353 seconds with half precision (BF16) 
 
 on a NVIDIA Blackwell GB203 GPU. 
 
 Therefore, on average  559.00 texts were classified per second. 
 
 

At batch size 4 a total of 1250  batches were created. 
 
 This task took  7.2180 seconds with half precision (BF16) 
 
 on a NVIDIA Blackwell GB203 GPU. 
 
 Therefore, on average  692.00 texts were classified per second. 
 
 

At batch size 8 a total of 625  batches were created. 
 
 This task took  6.7446 seconds with half precision (BF16) 
 
 on a NVIDIA Blackwell GB203 GPU. 
 
 Therefore, on average  741.00 texts were classified per second. 
 
 

At batch size 16 a total of 313  batches were created. 
 
 This task took  6.5108 seconds with half precision (BF16) 
 
 on a NVIDIA Blackwell GB203 GPU. 
 
 Therefore, on average  767.00 texts were classified per second. 
 
 

At batch size 32 a total of 157  batches were created. 
 
 This task took  6.4066 seconds

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


At batch size 1024 a total of 5  batches were created. 
 
 This task took  6.0849 seconds with half precision (BF16) 
 
 on a NVIDIA Blackwell GB203 GPU. 
 
 Therefore, on average  821.00 texts were classified per second. 
 
 

Note: At batch size 2048 the GPU ran out of memory.


<br><br><br>

In [7]:
#look at timings
df_history_bf16.head()

,batch_size,time
0,2,8.935320
1,4,7.218015
2,8,6.744613
3,16,6.510821
4,32,6.406609


In [8]:
print(f"Delta_max_bf16 =", df_history_bf16["time"].max() - df_history_bf16["time"].min())
print(f"Delta_2_1024_bf16 =", df_history_bf16["time"][0] - df_history_bf16["time"][9])

Delta_max_bf16 = 2.886431979999543
Delta_2_1024_bf16 = 2.850450534000629


<br><br><br>

# Batching at FP32 (GPU)

In [6]:
from transformers import pipeline
import pandas as pd
import torch
from time import perf_counter
from sklearn.metrics import f1_score, classification_report

#load pipeline
pipe = pipeline(
    task = cfg.task,
    model = cfg.model,
    device = cfg.device,
    dtype = torch.float32 #single precision
)

#different batch sizes
batches = [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]

#used for logging
history = []

for batch in batches:

    #number of batches
    if len(test_texts_final) % batch == 0:
        batch_number = len(test_texts_final)//batch
    else:
        batch_number = (len(test_texts_final)//batch) + 1
    
    #run model if it fits in video memory
    try:

        #start performance measuring
        start = perf_counter()
    
        results = pipe(
            test_texts_final,
            batch_size = batch,
            truncation = cfg.trunc
        )

        #stop time measurement                    
        end = perf_counter()

        #write results to dataframe
        df_results = pd.DataFrame(results)

        #recode label
        df_results["label"] = df_results["label"].replace(
            ["POSITIVE",
            "NEGATIVE"],
    
            [1,
            0]
        )
        

        #log times
        history.append({
            "batch_size": batch,
            "time": end - start})

        print(f"At batch size", batch, f"a total of", batch_number, f" batches were created. \n \n", 
              f"This task took {end - start: .4f} seconds with single precision (FP32) \n \n on a NVIDIA Blackwell GB203 GPU. \n \n Therefore, on average {len(test_texts_final)//(end - start): .2f} texts were classified per second. \n \n \n")

    
    #if video memory runs out, print explanation and then stop
    except torch.cuda.OutOfMemoryError:
        
        print(f"Note: At batch size", batch, f"the GPU ran out of memory.")
        
        break



#build dataframe with timings
df_history_fp32 = pd.DataFrame(history)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

At batch size 2 a total of 2500  batches were created. 
 
 This task took  17.1563 seconds with single precision (FP32) 
 
 on a NVIDIA Blackwell GB203 GPU. 
 
 Therefore, on average  291.00 texts were classified per second. 
 
 

At batch size 4 a total of 1250  batches were created. 
 
 This task took  15.4212 seconds with single precision (FP32) 
 
 on a NVIDIA Blackwell GB203 GPU. 
 
 Therefore, on average  324.00 texts were classified per second. 
 
 

At batch size 8 a total of 625  batches were created. 
 
 This task took  15.3928 seconds with single precision (FP32) 
 
 on a NVIDIA Blackwell GB203 GPU. 
 
 Therefore, on average  324.00 texts were classified per second. 
 
 

At batch size 16 a total of 313  batches were created. 
 
 This task took  15.1498 seconds with single precision (FP32) 
 
 on a NVIDIA Blackwell GB203 GPU. 
 
 Therefore, on average  330.00 texts were classified per second. 
 
 

At batch size 32 a total of 157  batches were created. 
 
 This task took  15

<br><br><br>

In [7]:
print(f"Delta_max_fp32 =", df_history_fp32["time"].max() - df_history_fp32["time"].min())
print(f"Delta_2_512_fp32 =", df_history_fp32["time"][0] - df_history_fp32["time"][8])

Delta_max_fp32 = 2.7278387429996656
Delta_2_512_fp32 = 2.3995907720000105


<br><br><br>

# Batching at FP32 (CPU)

In [15]:
from transformers import pipeline
import pandas as pd
import torch
from time import perf_counter
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split


#reduce dataset size because inferencing over 5000 texts via CPU will take forever

test_texts = list(raw_dataset["test"][cfg.texts])
test_labels = list(raw_dataset["test"][cfg.labels])

# Since we don't need the larger dataset, we can ignore its output by assigning it to `_` (nothing)
_, test_texts_final_cpu, _, test_labels_final_cpu = train_test_split(
    test_texts, test_labels,  # the data to split
    test_size = 768,    # the number of samples in the new dataset
    stratify = test_labels,      # to make sure the split is stratified based on the labels
    random_state = cfg.fixed_seed
)

#load pipeline
pipe = pipeline(
    task = cfg.task,
    model = cfg.model,
    device = torch.device("cpu"), #use cpu for test
    dtype = torch.float32 #single precision
)

#specify 16 cpu cores available
torch.set_num_threads(16)

#different batch sizes
#use smaller batches for cpu
batches = [16, 32, 64, 128, 256]

#used for logging
history = []

for batch in batches:

    #number of batches
    if len(test_texts_final_cpu) % batch == 0:
        batch_number = len(test_texts_final_cpu)//batch
    else:
        batch_number = (len(test_texts_final_cpu)//batch) + 1
    
    #run model if it fits in video memory
    try:

        #start performance measuring
        start = perf_counter()
    
        results = pipe(
            test_texts_final_cpu,
            batch_size = batch,
            truncation = cfg.trunc
        )

        #stop time measurement                    
        end = perf_counter()

        #write results to dataframe
        df_results = pd.DataFrame(results)

        #recode label
        df_results["label"] = df_results["label"].replace(
            ["POSITIVE",
            "NEGATIVE"],
    
            [1,
            0]
        )
        

        #log times
        history.append({
            "batch_size": batch,
            "time": end - start})

        print(f"At batch size", batch, f"a total of", batch_number, f" batches were created. \n \n", 
              f"This task took {end - start: .4f} seconds with single precision (FP32) \n \n on an Intel 12th Gen 12600K CPU @ 4.90 Gigahertz. \n \n Therefore, on average {len(test_texts_final_cpu)//(end - start): .2f} texts were classified per second. \n \n \n")

    
    #if memory runs out, print explanation and then stop
    except torch.cuda.OutOfMemoryError:
        
        print(f"Note: At batch size", batch, f"the system ran out of memory.")
        
        break



#build dataframe with timings
df_history_fp32_cpu = pd.DataFrame(history)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

At batch size 16 a total of 48  batches were created. 
 
 This task took  90.9271 seconds with single precision (FP32) 
 
 on an Intel 12th Gen 12600K CPU @ 4.90 Gigahertz. 
 
 Therefore, on average  8.00 texts were classified per second. 
 
 

At batch size 32 a total of 24  batches were created. 
 
 This task took  88.0251 seconds with single precision (FP32) 
 
 on an Intel 12th Gen 12600K CPU @ 4.90 Gigahertz. 
 
 Therefore, on average  8.00 texts were classified per second. 
 
 

At batch size 64 a total of 12  batches were created. 
 
 This task took  79.1438 seconds with single precision (FP32) 
 
 on an Intel 12th Gen 12600K CPU @ 4.90 Gigahertz. 
 
 Therefore, on average  9.00 texts were classified per second. 
 
 

At batch size 128 a total of 6  batches were created. 
 
 This task took  73.5766 seconds with single precision (FP32) 
 
 on an Intel 12th Gen 12600K CPU @ 4.90 Gigahertz. 
 
 Therefore, on average  10.00 texts were classified per second. 
 
 

At batch size 256 a

In [16]:
print(f"Delta_max_fp32_cpu =", df_history_fp32_cpu["time"].max() - df_history_fp32_cpu["time"].min())
print(f"Delta_2_512_fp32_cpu =", df_history_fp32_cpu["time"][0] - df_history_fp32_cpu["time"][3])

Delta_max_fp32_cpu = 21.94685778999701
Delta_2_512_fp32_cpu = 17.35049703599725
